# Quick Adapted Model Depth Preview

Minimal notebook for quickly visualizing **your adapted model's depth predictions** on a few images from selected datasets.

- No training
- No baseline comparisons
- Just RGB input + adapted depth output

In [ ]:
# Optional if needed in Kaggle runtime:
# !pip -q install -U datasets transformers matplotlib pillow tqdm

import os
import sys
from pathlib import Path
from typing import Any

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from datasets import load_dataset

# EagleVision imports (repo must be available)

In [ ]:
CONFIG = {
    # Kaggle paths
    "adapted_checkpoint_path": "/kaggle/input/models/rooonfr/best-adpated-100ep-scannet/pytorch/default/1/best_100ep.pt",
    "dav2_checkpoint_path": "baseline/depth_anything_v2/checkpoints/depth_anything_v2_metric_hypersim_vits.pth",

    # model settings
    "depth_mode": "metric",
    "encoder": "vits",
    "profile": "hypersim",
    "adapter_hidden_channels": 32,
    "normalize_backbone_input": False,

    # preview settings
    "samples_per_dataset": 3,
    "save_outputs": True,
    "output_dir": "outputs/quick_depth_preview",

    # datasets to sample from (same family you used)
    "datasets": [
        {"name": "nyu_depth_v2_hf", "type": "hf_nyu", "split": "validation", "enabled": True},
        {"name": "cifar100_val", "type": "hf_rgb", "hf_dataset": "cifar100", "split": "test", "image_key": "img", "enabled": True},
        {"name": "beans_val", "type": "hf_rgb", "hf_dataset": "beans", "split": "train", "image_key": "image", "enabled": True},
        {"name": "food101_val", "type": "hf_rgb", "hf_dataset": "food101", "split": "validation", "image_key": "image", "enabled": True},
        # optional local scannet-style samples
        {"name": "kaggle_scannet_2d", "type": "local_scannet_style", "root": "/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d", "enabled": False},
    ],
}

assert CONFIG["adapted_checkpoint_path"]
assert CONFIG["dav2_checkpoint_path"]

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

KAGGLE_WORKING = Path("/kaggle/working")
RUNNING_ON_KAGGLE = KAGGLE_WORKING.exists()
REPO_DIR = (KAGGLE_WORKING / "EagleVision") if (RUNNING_ON_KAGGLE and (KAGGLE_WORKING / "EagleVision").exists()) else Path.cwd().resolve()

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from eaglevision.models.depth.depth_anything_wrapper import DepthAnythingWithAdapter
from eaglevision.models.rt_depthnvs import RoundTripDepthNVS
from eaglevision.engine.checkpointing import load_checkpoint

OUT_DIR = REPO_DIR / CONFIG["output_dir"]
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("repo:", REPO_DIR)
print("output:", OUT_DIR)

In [ ]:
def resolve_ckpt(path_str: str, default_name: str | None = None) -> Path:
    p = Path(path_str)
    if not p.is_absolute():
        p = (REPO_DIR / p).resolve()
    if p.is_file():
        return p
    if p.is_dir():
        if default_name:
            c = p / default_name
            if c.is_file():
                return c
        cands = sorted(list(p.rglob("*.pth")) + list(p.rglob("*.pt")))
        if cands:
            return cands[0]
    raise FileNotFoundError(f"Checkpoint not found: {p}")


dav2_ckpt = resolve_ckpt(
    CONFIG["dav2_checkpoint_path"],
    default_name=f"depth_anything_v2_metric_{CONFIG['profile']}_{CONFIG['encoder']}.pth",
)
adapted_ckpt = resolve_ckpt(CONFIG["adapted_checkpoint_path"], default_name="best_100ep.pt")

print("dav2:", dav2_ckpt)
print("adapted:", adapted_ckpt)

In [ ]:
# Build adapted model once
depth_model = DepthAnythingWithAdapter(
    mode=CONFIG["depth_mode"],
    encoder=CONFIG["encoder"],
    profile=CONFIG["profile"],
    checkpoint_path=dav2_ckpt,
    freeze_backbone=True,
    adapter_hidden_channels=CONFIG["adapter_hidden_channels"],
    normalize_backbone_input=CONFIG["normalize_backbone_input"],
).to(DEVICE).eval()

model = RoundTripDepthNVS(depth_model).to(DEVICE).eval()
_ = load_checkpoint(adapted_ckpt, model)

print("Loaded adapted model checkpoint.")

In [ ]:
def to_tensor_rgb(image: Image.Image) -> torch.Tensor:
    arr = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1)


def robust_range(depth: np.ndarray):
    m = np.isfinite(depth)
    if m.sum() == 0:
        return 0.0, 1.0
    lo, hi = np.percentile(depth[m], [2, 98])
    if hi <= lo:
        hi = lo + 1e-6
    return float(lo), float(hi)


@torch.no_grad()
def predict_adapted_depth(image: Image.Image) -> np.ndarray:
    x = to_tensor_rgb(image).unsqueeze(0).to(DEVICE)
    d = model.depth_model(x)["adapted_depth"]
    if d.ndim == 4:      # [B,1,H,W]
        d = d[:, 0]
    if d.ndim != 3:      # [B,H,W]
        raise ValueError(f"Unexpected adapted_depth shape: {tuple(d.shape)}")
    d = d[0].detach().cpu().numpy().astype(np.float32)
    return d

In [ ]:
def load_hf_nyu_images(split: str, n: int):
    ds = None
    errors = []
    for cand in [
        {"path": "sayakpaul/nyu_depth_v2", "kwargs": {"revision": "refs/convert/parquet"}},
        {"path": "sayakpaul/nyu_depth_v2", "kwargs": {}},
    ]:
        try:
            ds = load_dataset(cand["path"], split=split, **cand["kwargs"])
            break
        except Exception as e:
            errors.append(str(e))
    if ds is None:
        raise RuntimeError("NYU load failed: " + " | ".join(errors))

    out = []
    for i in range(min(n, len(ds))):
        ex = ds[i]
        out.append({"dataset": "nyu_depth_v2_hf", "sample_id": f"nyu_{i:06d}", "image": ex["image"].convert("RGB")})
    return out


def load_hf_rgb_images(dataset_name: str, split: str, image_key: str, n: int, label: str):
    ds = load_dataset(dataset_name, split=split)
    out = []
    for i in range(min(n, len(ds))):
        im = ds[i][image_key]
        if not isinstance(im, Image.Image):
            im = Image.fromarray(np.asarray(im))
        out.append({"dataset": label, "sample_id": f"{label}_{i:06d}", "image": im.convert("RGB")})
    return out


def load_local_scannet_images(root: str, n: int):
    rootp = Path(root)
    rows = []
    if not rootp.exists():
        return rows
    scenes = sorted([p for p in rootp.glob("*") if p.is_dir()])
    for scene in scenes:
        cdir = next((d for d in [scene / "color", scene / "rgb", scene / "images"] if d.exists()), None)
        if cdir is None:
            continue
        for cp in sorted(cdir.glob("*.jpg")) + sorted(cdir.glob("*.png")):
            rows.append({"dataset": "kaggle_scannet_2d", "sample_id": f"{scene.name}_{cp.stem}", "image": Image.open(cp).convert("RGB")})
            if len(rows) >= n:
                return rows
    return rows

In [ ]:
# Collect a few images from each enabled dataset
all_samples = []
n = int(CONFIG["samples_per_dataset"])

for d in CONFIG["datasets"]:
    if not d.get("enabled", False):
        continue
    try:
        if d["type"] == "hf_nyu":
            s = load_hf_nyu_images(d.get("split", "validation"), n)
        elif d["type"] == "hf_rgb":
            s = load_hf_rgb_images(d["hf_dataset"], d.get("split", "train"), d.get("image_key", "image"), n, d["name"])
        elif d["type"] == "local_scannet_style":
            s = load_local_scannet_images(d["root"], n)
        else:
            s = []
        print(f"{d['name']}: {len(s)} samples")
        all_samples.extend(s)
    except Exception as e:
        print(f"[skip dataset] {d['name']}: {e}")

print("Total preview samples:", len(all_samples))

In [ ]:
# Predict + visualize inline
if len(all_samples) == 0:
    raise RuntimeError("No samples loaded. Check dataset config and paths.")

for sample in all_samples:
    img = sample["image"]
    sid = sample["sample_id"]
    dname = sample["dataset"]

    depth = predict_adapted_depth(img)
    vmin, vmax = robust_range(depth)

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].imshow(np.asarray(img))
    ax[0].set_title(f"RGB | {dname} | {sid}")
    ax[0].axis("off")

    ax[1].imshow(depth, cmap="magma", vmin=vmin, vmax=vmax)
    ax[1].set_title(f"Adapted Depth | range [{vmin:.3f}, {vmax:.3f}]")
    ax[1].axis("off")
    plt.tight_layout()
    plt.show()

    if CONFIG["save_outputs"]:
        out_img = OUT_DIR / f"{dname}__{sid}__depth.png"
        # Save colorized map
        norm = np.clip((depth - vmin) / max(vmax - vmin, 1e-6), 0, 1)
        color = (plt.get_cmap("magma")(norm)[..., :3] * 255).astype(np.uint8)
        Image.fromarray(color).save(out_img)

print("Done. Saved outputs to:", OUT_DIR)